# Bulgaria (Santen) — Monthly Processing
## Step 1: Load raw input → Data_Original → Data → Dataraw_df

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

### Parameters
Set `INPUT_PATH` to the full path of this month's raw file (`BG MM YY.xlsx` or `BG MM YYYY.xlsx`).

In [ ]:
INPUT_PATH = r"D:\Bulgaria\<month folder>\BG MM YY.xlsx"   # <-- set each month

MAIN_SHEET = 0   # main data sheet selected by position (first sheet)

### Parse month/year from filename
Accepts spaces or underscores as separators, and a 2- or 4-digit year.

In [ ]:
path = Path(INPUT_PATH)
m = re.match(r"^BG[ _](\d{2})[ _](\d{2}|\d{4})$", path.stem, flags=re.IGNORECASE)
if not m:
    raise ValueError(f"Filename '{path.name}' does not match 'BG MM YY' / 'BG MM YYYY'")

proc_month = int(m.group(1))
proc_year = int(m.group(2)) + (2000 if len(m.group(2)) == 2 else 0)
if not 1 <= proc_month <= 12:
    raise ValueError(f"Invalid month '{m.group(1)}' in filename")

PROCESS_MONTH = pd.Timestamp(year=proc_year, month=proc_month, day=1)
print("File          :", path.name)
print("Process month :", PROCESS_MONTH.strftime("%b-%Y"))

### Load main data sheet → Data_Original (frozen reference)

In [ ]:
xl = pd.ExcelFile(path)
print("Main sheet (position 0):", xl.sheet_names[MAIN_SHEET])

Data_Original = pd.read_excel(xl, sheet_name=MAIN_SHEET)
_orig_hash = pd.util.hash_pandas_object(Data_Original, index=True).sum()   # used to confirm it is never modified

print("Shape   :", Data_Original.shape)
print("Columns :", list(Data_Original.columns))

### Check: latest month in data = month in filename

In [ ]:
time_col = Data_Original["Time"]
print("Time range :", time_col.min().strftime("%b-%Y"), "→", time_col.max().strftime("%b-%Y"),
      f"({time_col.nunique()} months)")

if time_col.max() != PROCESS_MONTH:
    raise ValueError(f"Latest month in data ({time_col.max():%b-%Y}) "
                     f"≠ filename month ({PROCESS_MONTH:%b-%Y})")
print("OK: latest month matches filename")

### Working copy

In [ ]:
Data = Data_Original.copy()

### Changes to Data
**1. Drop `Pack`** (by name; stops if missing)

In [ ]:
if "Pack" not in Data.columns:
    raise KeyError(f"'Pack' column not found. Columns: {list(Data.columns)}")
Data = Data.drop(columns="Pack")
print("Dropped 'Pack' →", Data.shape)

**2. Rename headers**  
Columns 1, 2, 5–9: exact header check, then rename.  
Columns 3–4: renamed by position (header text may vary month to month).

In [ ]:
# (position, expected header or None if it may vary, new header)
RENAME_MAP = [
    (0, "Santen 2023",    "Region - Level 0"),
    (1, "Region Level 2", "Region - Level 1"),
    (2, None,             "Product - Level 1"),   # e.g. 'EPHMRA ATC 2'
    (3, None,             "Product - Level 2"),   # e.g. 'EPHMRA ATC 3'
    (4, "Corporation",    "Product - Level 3"),
    (5, "Product",        "Product - Level 4"),
    (6, "Time",           "Time - Level 2"),
    (7, "Metric",         "Measure"),
    (8, "Value",          "VALUE"),
]

found = list(Data.columns)
problems = []
if len(found) != len(RENAME_MAP):
    problems.append(f"expected {len(RENAME_MAP)} columns, found {len(found)}")
for pos, expected, _ in RENAME_MAP:
    got = found[pos] if pos < len(found) else "<missing>"
    if expected is not None and got != expected:
        problems.append(f"col {pos+1}: expected '{expected}', found '{got}'")

if problems:
    print(pd.DataFrame({"position": range(1, len(RENAME_MAP)+1),
                        "expected": [e if e else "(any)" for _, e, _ in RENAME_MAP],
                        "found": [found[p] if p < len(found) else "<missing>" for p, _, _ in RENAME_MAP]}).to_string(index=False))
    raise ValueError("Header check failed:\n  " + "\n  ".join(problems))

Data.columns = [new for _, _, new in RENAME_MAP]
print("Columns 3–4 were:", found[2], "|", found[3])
print("Renamed columns :", list(Data.columns))

### Final

In [ ]:
Dataraw_df = Data

assert pd.util.hash_pandas_object(Data_Original, index=True).sum() == _orig_hash, "Data_Original was modified!"

# Validation vs Data_Original: row count and VALUE sum by Measure must be identical
chk = pd.DataFrame({
    "Data_Original": Data_Original.groupby("Metric")["Value"].sum(),
    "Dataraw_df":    Dataraw_df.groupby("Measure")["VALUE"].sum(),
})
chk["diff"] = chk["Dataraw_df"] - chk["Data_Original"]
print(f"Rows: Data_Original={len(Data_Original):,}  Dataraw_df={len(Dataraw_df):,}")
print(chk.to_string(float_format="{:,.2f}".format))
assert len(Dataraw_df) == len(Data_Original) and (chk["diff"].abs() < 1e-6).all(), "Validation failed"
print("\nOK: Data_Original unchanged", Data_Original.shape, "| Dataraw_df", Dataraw_df.shape)

---
## Step 2: Dataraw_df + mapping file → working_model_df → working_bg_df (pivot)

### Parameters (Step 2)

In [ ]:
MAPPING_PATH = r"D:\Bulgaria\<month folder>\Mapping Document_Bulgaria.xlsx"   # <-- set each month

# Output Working file (default: same folder as the raw file)
OUTPUT_PATH = str(Path(INPUT_PATH).with_name(f"Working File BG {PROCESS_MONTH:%m %Y}.xlsx"))

# Optional: a finished Working file to validate against (None = skip validation)
REFERENCE_WORKING_FILE = None

### 2.1 Read mapping sheets (each into its own dataframe)

In [ ]:
xl_map = pd.ExcelFile(MAPPING_PATH)
REQUIRED = {
    "ATC & Category":   ["ATC Class", "Category"],
    "Category mapping": ["Product", "Brand", "Corporation", "Company", "International Pack", "HU conversion factor",
                         "ATC class", "Category", "Sub-market", "Formulation", "Molecule", "Ingredient", "Parallel import"],
    "Reps":             ["Reps", "RSM", "GM/CM/NSM"],
    "Bricks":           ["Brick", "Region", "Territory", "RSM"],
}
missing_sheets = [s for s in REQUIRED if s not in xl_map.sheet_names]
if missing_sheets:
    raise ValueError(f"Mapping file is missing sheets: {missing_sheets}. Found: {xl_map.sheet_names}")

# All text, as in Power Query; HU conversion factor converted to number below
atc_category_df     = pd.read_excel(xl_map, sheet_name="ATC & Category",   dtype=str)
category_mapping_df = pd.read_excel(xl_map, sheet_name="Category mapping", dtype=str)
reps_df             = pd.read_excel(xl_map, sheet_name="Reps",             dtype=str)
bricks_df           = pd.read_excel(xl_map, sheet_name="Bricks",           dtype=str)

for name, df in {"ATC & Category": atc_category_df, "Category mapping": category_mapping_df,
                 "Reps": reps_df, "Bricks": bricks_df}.items():
    miss = [c for c in REQUIRED[name] if c not in df.columns]
    if miss:
        raise KeyError(f"'{name}' is missing columns {miss}. Found: {list(df.columns)}")
    print(f"{name:17s} {df.shape} -> {list(df.columns)}")

hu_raw = category_mapping_df["HU conversion factor"]
category_mapping_df["HU conversion factor"] = pd.to_numeric(hu_raw, errors="coerce")
bad_hu = hu_raw.notna() & category_mapping_df["HU conversion factor"].isna()
if bad_hu.any():
    print(f"WARNING: {bad_hu.sum()} non-numeric 'HU conversion factor' values set to blank:",
          hu_raw[bad_hu].unique().tolist())

### 2.2 Prepare Category mapping (Power Query: remove blank products → Key → keep first per Key)

In [ ]:
cm = category_mapping_df.copy()
cm = cm[cm["Product"].notna() & (cm["Product"].str.strip() != "")]
cm["Key"] = cm["Product"].str.strip() + cm["Corporation"].fillna("").str.strip()

dup = cm[cm["Key"].duplicated(keep=False)]
if len(dup):
    print(f"Duplicate Keys in Category mapping (first row kept, as in Power Query): {dup['Key'].nunique()}")
    print(dup[["Key", "Brand", "Company", "International Pack", "Category"]].to_string())
cm = cm.drop_duplicates("Key", keep="first")
print("Category mapping rows used:", len(cm))

### 2.3 Build working_model_df
Pivot Measure → Units / Value MNF (EUR); add Key; join Category mapping; add Data Model columns.

In [ ]:
DIMS = ["Region - Level 0", "Region - Level 1", "Product - Level 1", "Product - Level 2",
        "Product - Level 3", "Product - Level 4", "Time - Level 2"]
MEASURES = ["Units", "Value MNF (EUR)"]

found_measures = sorted(Dataraw_df["Measure"].dropna().unique())
if found_measures != sorted(MEASURES):
    raise ValueError(f"Unexpected Measure values: {found_measures} (expected {MEASURES})")

# Pivot Measure into columns, summing VALUE (Table.Pivot ... List.Sum)
model = (Dataraw_df.groupby(DIMS + ["Measure"], dropna=False)["VALUE"].sum(min_count=1)
                   .unstack("Measure").reset_index())
model.columns.name = None
model = model[DIMS + MEASURES]

# Key = Trim(Product - Level 4) & Trim(Product - Level 3)
model["Key"] = model["Product - Level 4"].str.strip() + model["Product - Level 3"].fillna("").str.strip()

# Left join Category mapping on Key (renamed as in the Working file)
CM_COLS = {"Brand": "Brand1", "Company": "Company1", "International Pack": "International pack",
           "HU conversion factor": "HU conversion factor", "ATC class": "ATC class", "Category": "Category1",
           "Sub-market": "Sub-market", "Formulation": "Formulation1", "Molecule": "Molecule1",
           "Ingredient": "Ingredient1", "Parallel import": "Parallel import"}
model = model.merge(cm[["Key"] + list(CM_COLS)].rename(columns=CM_COLS),
                    on="Key", how="left", validate="many_to_one")
print("Rows after pivot + mapping join:", len(model))

**New product check** — any blank Brand, Company or Category after the join

In [ ]:
def _is_blank(s):
    return s.isna() | (s.astype(str).str.strip() == "")

new_flag = _is_blank(model["Brand1"]) | _is_blank(model["Company1"]) | _is_blank(model["Category1"])
DETAIL_COLS = ["Product - Level 4", "Product - Level 3", "Product - Level 1", "Product - Level 2", "Key",
               "Brand1", "Company1", "Category1", "International pack", "HU conversion factor", "ATC class",
               "Sub-market", "Formulation1", "Molecule1", "Ingredient1", "Parallel import"]

new_products_df = pd.DataFrame(columns=DETAIL_COLS)
if new_flag.any():
    rows = []
    for key, grp in model[new_flag].groupby("Key", dropna=False, sort=True):
        r = grp.iloc[0]
        details = " | ".join(f"{c}: {'(blank)' if _is_blank(pd.Series([r[c]]))[0] else r[c]}" for c in DETAIL_COLS)
        print(f"New product detected. Details: {details}")
        print(f"    affects {len(grp)} rows | Units {grp['Units'].sum():,.0f} | Value MNF (EUR) {grp['Value MNF (EUR)'].sum():,.2f} | "
              f"months {grp['Time - Level 2'].min():%b-%Y} to {grp['Time - Level 2'].max():%b-%Y}\n")
        rows.append(r[DETAIL_COLS])
    new_products_df = pd.DataFrame(rows).reset_index(drop=True)
else:
    print("No new products detected (Brand, Company and Category populated for all rows).")

**Data Model calculated columns** (decoded from the Working file's Data Model)

In [ ]:
# category = RELATED('Category mapping'[Category])
model["category"] = model["Category1"]

# Retina Market = IF(IFERROR(FIND("Retina",[category]),0)=0,"Exclude","Include")   (FIND is case-sensitive)
model["Retina Market"] = np.where(model["category"].fillna("").str.contains("Retina", regex=False),
                                  "Include", "Exclude")

# Region / RSM / Territory / Brick = RELATED(Bricks[...]) via Region - Level 1 = Brick
br = bricks_df[["Brick", "Region", "RSM", "Territory"]].copy()
br["_brick_key"] = br["Brick"]
model = model.merge(br, left_on="Region - Level 1", right_on="_brick_key",
                    how="left", validate="many_to_one")

# New brick check — Region - Level 1 not found in the Bricks sheet
no_brick = model["_brick_key"].isna()
new_bricks_df = pd.DataFrame(columns=["Region - Level 1", "Region - Level 0"])
if no_brick.any():
    for brick, grp in model[no_brick].groupby("Region - Level 1", dropna=False, sort=True):
        print(f"New brick detected. Details: Region - Level 1: {brick} | "
              f"Region - Level 0: {', '.join(grp['Region - Level 0'].dropna().unique())}")
        print(f"    affects {len(grp)} rows | Units {grp['Units'].sum():,.0f} | "
              f"Value MNF (EUR) {grp['Value MNF (EUR)'].sum():,.2f}\n")
    new_bricks_df = (model.loc[no_brick, ["Region - Level 1", "Region - Level 0"]]
                          .drop_duplicates().reset_index(drop=True))
else:
    print("No new bricks detected (every Region - Level 1 found in the Bricks sheet).")
model = model.drop(columns="_brick_key")

# Remaining RELATED('Category mapping'[...]) columns
model["Brand"]       = model["Brand1"]
model["Company"]     = model["Company1"]
model["Pack"]        = model["International pack"]
model["Molecule"]    = model["Molecule1"]
model["Sub Market"]  = model["Sub-market"]
model["Formulation"] = model["Formulation1"]
model["Ingredient"]  = model["Ingredient1"]
model["Import"]      = model["Parallel import"]

# HU_Units = RELATED('Category mapping'[HU conversion factor]) * Data[Units]
model["HU_Units"] = model["HU conversion factor"] * model["Units"]

MODEL_COLUMNS = DIMS + MEASURES + ["Key"] + list(CM_COLS.values()) + [
    "category", "Retina Market", "Region", "RSM", "Territory", "Brick", "Brand", "Company", "Pack",
    "Molecule", "Sub Market", "Formulation", "Ingredient", "Import", "HU_Units"]
working_model_df = model[MODEL_COLUMNS]
print("working_model_df:", working_model_df.shape)

### 2.4 Build working_bg_df (the pivot on the Working file's `Data` sheet)
Rows: 16 fields · Columns: Sum of HU_Units × months, then Sum of Value MNF (EUR) × months · no subtotals/grand totals · blank where no data.

In [ ]:
ROW_FIELDS = ["Retina Market", "Region", "RSM", "Territory", "Brick", "Product - Level 1", "Category1",
              "Product - Level 4", "Brand1", "Company1", "International pack", "Molecule1", "Sub-market",
              "Formulation1", "Ingredient1", "Parallel import"]
VALUE_FIELDS = {"HU_Units": "Sum of HU_Units", "Value MNF (EUR)": "Sum of Value MNF (EUR)"}

months = sorted(working_model_df["Time - Level 2"].dropna().unique())   # dynamic: every month in the data

wide = (working_model_df.groupby(ROW_FIELDS + ["Time - Level 2"], dropna=False)[list(VALUE_FIELDS)]
                        .sum(min_count=1)
                        .unstack("Time - Level 2"))
wide = wide.rename(columns=VALUE_FIELDS, level=0)
wide = wide.reindex(columns=pd.MultiIndex.from_product([list(VALUE_FIELDS.values()), months]))
# Row order as Excel's pivot: A→Z per field, case-insensitive, ignoring hyphens and apostrophes
_excel_sort_key = lambda lvl: lvl.map(lambda v: v if pd.isna(v) else str(v).lower().replace("-", "").replace("'", ""))
wide = wide.sort_index(key=_excel_sort_key, na_position="last", sort_remaining=False)
working_bg_df = wide.reset_index()

print("working_bg_df:", working_bg_df.shape, f"| months {pd.Timestamp(months[0]):%b-%Y} to {pd.Timestamp(months[-1]):%b-%Y} ({len(months)})")

### 2.5 Write the Working file (same 3 header rows as the pivot)

In [ ]:
n_lab, n_m = len(ROW_FIELDS), len(months)
hdr1 = [None] * n_lab + ["Values", "Time - Level 2"] + [None] * (2 * n_m - 2)
hdr2 = [None] * n_lab + ["Sum of HU_Units"] + [None] * (n_m - 1) + ["Sum of Value MNF (EUR)"] + [None] * (n_m - 1)
hdr3 = ROW_FIELDS + [pd.Timestamp(m) for m in months] * 2

body = working_bg_df.astype(object).where(working_bg_df.notna(), None).values.tolist()
out = pd.DataFrame([hdr1, hdr2, hdr3] + body)

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl", datetime_format="m/d/yyyy", date_format="m/d/yyyy") as xw:
    out.to_excel(xw, sheet_name="Data", header=False, index=False)
print("Written:", OUTPUT_PATH, "| sheet 'Data' |", out.shape[0], "rows x", out.shape[1], "cols")

### 2.6 Validation vs reference Working file (optional, order-independent)

In [ ]:
def _read_pivot(path):
    p = pd.read_excel(path, sheet_name="Data", header=None)
    header = p.iloc[:3].reset_index(drop=True)
    labels = p.iloc[3:, :16].reset_index(drop=True)
    values = p.iloc[3:, 16:].astype(float).reset_index(drop=True)
    return header, labels, values

if REFERENCE_WORKING_FILE:
    # (a) working_model_df vs reference Sheet1 (model table)
    ref_model = pd.read_excel(REFERENCE_WORKING_FILE, sheet_name="Sheet1")
    print("Model table vs reference Sheet1")
    print(f"  columns identical & same order: {list(ref_model.columns) == MODEL_COLUMNS}")
    print(f"  rows: ours {len(working_model_df):,} | ref {len(ref_model):,}")
    for c in ["Units", "Value MNF (EUR)", "HU_Units"]:
        print(f"  sum {c:16s}: ours {working_model_df[c].sum():,.2f} | ref {ref_model[c].sum():,.2f} | "
              f"diff {working_model_df[c].sum() - ref_model[c].sum():,.6f}")

    # (b) output file vs reference pivot (Data sheet)
    oh, ol, ov = _read_pivot(OUTPUT_PATH)
    rh, rl, rv = _read_pivot(REFERENCE_WORKING_FILE)
    col_sums = pd.DataFrame({"ours": ov.sum().values, "ref": rv.sum().values}) if ov.shape == rv.shape else None
    print("\nPivot (Data sheet) — output file vs reference")
    print(f"  3 header rows identical : {oh.equals(rh)}")
    print(f"  rows                    : ours {len(ol):,} | ref {len(rl):,}")
    print(f"  columns                 : ours {ol.shape[1] + ov.shape[1]} | ref {rl.shape[1] + rv.shape[1]}")
    print(f"  row-label combos match  : {set(map(tuple, ol.astype(str).values)) == set(map(tuple, rl.astype(str).values))}")
    print(f"  row order identical     : {ol.astype(str).equals(rl.astype(str))}")
    print(f"  blank value cells       : ours {int(ov.isna().sum().sum()):,} | ref {int(rv.isna().sum().sum()):,}")
    if col_sums is not None:
        col_sums["diff"] = col_sums["ours"] - col_sums["ref"]
        print(f"  {len(col_sums)} value-column sums: max abs diff {col_sums['diff'].abs().max():.6f}")
        print(f"  Sum of HU_Units total        : ours {ov.iloc[:, :n_m].sum().sum():,.2f} | ref {rv.iloc[:, :n_m].sum().sum():,.2f}")
        print(f"  Sum of Value MNF (EUR) total : ours {ov.iloc[:, n_m:].sum().sum():,.2f} | ref {rv.iloc[:, n_m:].sum().sum():,.2f}")
    print("\n(If row order ever differs from Excel's pivot, it is cosmetic — totals and row sets are what count.)")
else:
    print("No REFERENCE_WORKING_FILE set — validation skipped.")

---
## Step 3: working_bg_df → Country Dashboard `Data` sheet (dashboard_df)

### Parameters (Step 3)

In [ ]:
# Previous Country Dashboard Data file: header colours / column widths are copied from its 'Data' sheet
TEMPLATE_PATH  = r"D:\Bulgaria\<month folder>\Country Dashboard Data - Bulgaria.xlsx"   # <-- set each month
TEMPLATE_SHEET = "Data"

FY_START_MONTH = 4      # Bulgaria fiscal year starts in April
N_MONTHS       = 24     # trailing months shown in the UN / REV blocks
END_MONTH      = None   # None = latest month in the data; override e.g. "2026-07-01"

DASHBOARD_OUTPUT_PATH = str(Path(INPUT_PATH).with_name(f"Country Dashboard Data - Bulgaria {PROCESS_MONTH:%m %Y}.xlsx"))

# Optional: a finished dashboard file for the same month to validate against (None = skip)
REFERENCE_DASHBOARD_FILE = None

### 3.1 New product check (on working_bg_df) — report only

In [ ]:
db_labels = working_bg_df.iloc[:, :len(ROW_FIELDS)].copy()
db_labels.columns = ROW_FIELDS

_blank = lambda s: s.isna() | (s.astype(str).str.strip() == "")
new_mask = _blank(db_labels["Brand1"]) | _blank(db_labels["Company1"]) | _blank(db_labels["Category1"])
if new_mask.any():
    names = sorted(db_labels.loc[new_mask, "Product - Level 4"].dropna().astype(str).unique())
    print(f"New product detected: {', '.join(names)}")
else:
    print("no new products found")

### 3.2 Label columns, UN / REV month blocks

In [ ]:
DB_LABEL_MAP = {"Retina Market": "Retina market", "Region": "Region", "RSM": "RSM", "Territory": "Territory",
                "Brick": "Brick", "Category1": "Category", "Brand1": "Brand", "Company1": "Company",
                "International pack": "Pack", "Molecule1": "Molecule", "Sub-market": "Sub-Market",
                "Formulation1": "Formulation", "Ingredient1": "Ingredient", "Parallel import": "Import"}
db_lab = db_labels[list(DB_LABEL_MAP)].rename(columns=DB_LABEL_MAP)      # drops Product - Level 1 / Level 4

end_month = pd.Timestamp(END_MONTH) if END_MONTH else pd.Timestamp(max(months))
db_months = list(pd.date_range(end=end_month, periods=N_MONTHS, freq="MS"))

missing = [m for m in db_months if m not in set(pd.Timestamp(x) for x in months)]
if missing:
    print("WARNING: months with no data at all (filled with 0):", [f"{m:%b %Y}" for m in missing])

def _block(measure_label):
    b = working_bg_df[measure_label].copy()
    b.columns = [pd.Timestamp(c) for c in b.columns]
    return b.reindex(columns=db_months).fillna(0)          # blank months -> 0

un_blk  = _block("Sum of HU_Units")         # UN  = HU units (Units x HU conversion factor)
rev_blk = _block("Sum of Value MNF (EUR)")  # REV = Value MNF (EUR)
print(f"Months: {db_months[0]:%b %Y} -> {db_months[-1]:%b %Y} ({len(db_months)})")

### 3.3 Metric columns
Per measure (UN, REV) and period (Current = end month, Previous Period = −1 month, Previous Year = −12 months):  
single month · Fixed Qtr (fiscal QTD) · Rolling Qtr (3 m) · Fixed Tercile (fiscal 4-month block to date: Apr–Jul, Aug–Nov, Dec–Mar) · Rolling Tercile (4 m) · FY YTD · MAT (12 m)

In [ ]:
def period_metrics(blk, end):
    e = db_months.index(end)
    off = (end.month - FY_START_MONTH) % 12              # months since fiscal-year start
    s = lambda a: blk.iloc[:, a:e + 1].sum(axis=1)
    if e - 11 < 0:
        raise ValueError(f"Not enough months for MAT ending {end:%b %Y}")
    return {"": blk.iloc[:, e], "Fixed Qtr ": s(e - off % 3), "Rolling Qtr ": s(e - 2),
            "Fixed Tercile ": s(e - off % 4), "Rolling Tercile ": s(e - 3), "FY YTD ": s(e - off), "MAT ": s(e - 11)}

PERIODS = [("Current", end_month), ("Previous Period", end_month - pd.DateOffset(months=1)),
           ("Previous Year", end_month - pd.DateOffset(months=12))]

cols = {}
for meas, blk in [("UN", un_blk), ("REV", rev_blk)]:
    for m in db_months:
        cols[f"{meas}-{m:%b %Y}"] = blk[m]
for meas, blk in [("UN", un_blk), ("REV", rev_blk)]:
    for per, end in PERIODS:
        for name, ser in period_metrics(blk, end).items():
            cols[f"{per} {meas}-{name}{end:%b %Y}"] = ser

dashboard_df = pd.concat([pd.Series(None, index=db_lab.index, name="Key", dtype=object),
                          db_lab, pd.DataFrame(cols),
                          pd.Series(None, index=db_lab.index, name="Key 2", dtype=object)], axis=1)
print("dashboard_df:", dashboard_df.shape)

### 3.4 Write Data sheet (pandas), then copy header styles / widths from the template (openpyxl)

In [ ]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from copy import copy

with pd.ExcelWriter(DASHBOARD_OUTPUT_PATH, engine="openpyxl") as xw:
    dashboard_df.to_excel(xw, sheet_name="Data", index=False)

tpl_ws = load_workbook(TEMPLATE_PATH)[TEMPLATE_SHEET]
if tpl_ws.max_column != dashboard_df.shape[1]:
    raise ValueError(f"Template has {tpl_ws.max_column} columns, output has {dashboard_df.shape[1]}")

wb = load_workbook(DASHBOARD_OUTPUT_PATH)
ws = wb["Data"]
for i in range(1, dashboard_df.shape[1] + 1):
    src, dst = tpl_ws.cell(row=1, column=i), ws.cell(row=1, column=i)
    dst.font, dst.fill, dst.border = copy(src.font), copy(src.fill), copy(src.border)
    dst.alignment, dst.number_format, dst.protection = copy(src.alignment), src.number_format, copy(src.protection)
    # value stays = this month's label written by pandas (header rolled forward)

for dim in tpl_ws.column_dimensions.values():          # column widths (template stores ranges min..max)
    if dim.width and dim.min:
        for c in range(dim.min, min(dim.max, dashboard_df.shape[1]) + 1):
            ws.column_dimensions[get_column_letter(c)].width = dim.width
if tpl_ws.row_dimensions[1].height:
    ws.row_dimensions[1].height = tpl_ws.row_dimensions[1].height
wb.save(DASHBOARD_OUTPUT_PATH)

print("Written:", DASHBOARD_OUTPUT_PATH, "| sheet 'Data' |", dashboard_df.shape[0], "rows x", dashboard_df.shape[1], "cols")
print("Header row:", list(dashboard_df.columns[[15, 15 + N_MONTHS - 1, 63, 70, 77]]))

### 3.5 Validation vs reference dashboard file (optional, order-independent)

In [ ]:
if REFERENCE_DASHBOARD_FILE:
    ref = pd.read_excel(REFERENCE_DASHBOARD_FILE, sheet_name="Data")
    ours = pd.read_excel(DASHBOARD_OUTPUT_PATH, sheet_name="Data")
    lab_cols, num_cols = list(ours.columns[1:15]), list(ours.columns[15:-1])
    print(f"Column names identical & same order : {list(ours.columns) == list(ref.columns)}")
    print(f"Rows                                : ours {len(ours):,} | ref {len(ref):,}")
    print(f"Label rows match (as a set)         : "
          f"{sorted(map(tuple, ours[lab_cols].astype(str).values)) == sorted(map(tuple, ref[lab_cols].astype(str).values))}")
    if list(ours.columns) == list(ref.columns):
        cs = pd.DataFrame({"ours": ours[num_cols].sum(), "ref": ref[num_cols].apply(pd.to_numeric, errors="coerce").sum()})
        cs["diff"] = cs["ours"] - cs["ref"]
        bad = cs[cs["diff"].abs() > 1e-4]
        print(f"Per-column sums ({len(num_cols)} numeric cols)   : {len(num_cols) - len(bad)} match, {len(bad)} differ")
        if len(bad):
            print(bad.to_string(float_format="{:,.2f}".format))
else:
    print("No REFERENCE_DASHBOARD_FILE set — validation skipped.")